In [1]:
# =========================
# Customer Churn Prediction Project
# Colab-ready single-cell code
# =========================

# Install required libraries
!pip install -q xgboost imbalanced-learn

import pandas as pd
import numpy as np
import joblib
import json

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# =========================
# Step 1: Generate Synthetic Dataset
# =========================

def generate_customer_churn_data(n_samples=1000, random_state=42):
    rng = np.random.default_rng(random_state)

    tenure = rng.integers(1, 72, n_samples)
    monthly_charges = rng.normal(70, 25, n_samples).clip(20, 150)
    total_charges = (tenure * monthly_charges + rng.normal(0, 100, n_samples)).clip(20, None)
    contract_type = rng.choice(["Month-to-month", "One year", "Two year"], n_samples, p=[0.55, 0.25, 0.20])
    internet_service = rng.choice(["DSL", "Fiber optic", "No"], n_samples, p=[0.35, 0.50, 0.15])
    payment_method = rng.choice(
        ["Electronic check", "Mailed check", "Bank transfer", "Credit card"],
        n_samples,
        p=[0.35, 0.20, 0.20, 0.25]
    )
    support_calls = rng.poisson(2.0, n_samples)
    senior_citizen = rng.choice([0, 1], n_samples, p=[0.82, 0.18])
    partner = rng.choice([0, 1], n_samples, p=[0.45, 0.55])
    dependents = rng.choice([0, 1], n_samples, p=[0.70, 0.30])

    churn_score = (
        0.035 * monthly_charges
        - 0.03 * tenure
        + 0.25 * support_calls
        + 0.45 * senior_citizen
        - 0.35 * partner
        - 0.25 * dependents
        + np.where(contract_type == "Month-to-month", 1.0, 0.0)
        + np.where(internet_service == "Fiber optic", 0.8, 0.0)
        + np.where(payment_method == "Electronic check", 0.6, 0.0)
        - 2.2
    )

    churn_prob = 1 / (1 + np.exp(-churn_score))
    churn = rng.binomial(1, churn_prob)

    df = pd.DataFrame({
        "Tenure": tenure,
        "MonthlyCharges": np.round(monthly_charges, 2),
        "TotalCharges": np.round(total_charges, 2),
        "ContractType": contract_type,
        "InternetService": internet_service,
        "PaymentMethod": payment_method,
        "SupportCalls": support_calls,
        "SeniorCitizen": senior_citizen,
        "Partner": partner,
        "Dependents": dependents,
        "Churn": churn
    })

    return df

df = generate_customer_churn_data()
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nChurn distribution:")
print(df["Churn"].value_counts())

# Save dataset
df.to_csv("customer_churn_sample.csv", index=False)
print("\nDataset saved as customer_churn_sample.csv")

# =========================
# Step 2: Split Features and Target
# =========================

X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# Step 3: Preprocessing
# =========================

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# =========================
# Step 4: Define Models
# =========================

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=250,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42
    )
}

# =========================
# Step 5: Train and Evaluate
# =========================

results = {}
trained_models = {}

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1-Score": round(f1_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_prob), 4)
    }

    print(f"\n{'='*50}")
    print(f"{name} Results")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred))
    print("Metrics:", metrics)

    return metrics

for name, classifier in models.items():
    pipeline = ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)
    metrics = evaluate_model(name, pipeline, X_test, y_test)

    results[name] = metrics
    trained_models[name] = pipeline

# =========================
# Step 6: Compare Results
# =========================

results_df = pd.DataFrame(results).T
print("\nFinal Model Comparison:")
print(results_df)

# =========================
# Step 7: Save Best Model
# =========================

best_model_name = results_df["F1-Score"].idxmax()
best_model = trained_models[best_model_name]

joblib.dump(best_model, "best_churn_model.pkl")

with open("metrics.json", "w") as f:
    json.dump(results, f, indent=4)

print(f"\nBest Model: {best_model_name}")
print("Saved best model as best_churn_model.pkl")
print("Saved metrics as metrics.json")

# =========================
# Step 8: Test on New Customer
# =========================

sample_customer = pd.DataFrame([{
    "Tenure": 8,
    "MonthlyCharges": 95.20,
    "TotalCharges": 761.60,
    "ContractType": "Month-to-month",
    "InternetService": "Fiber optic",
    "PaymentMethod": "Electronic check",
    "SupportCalls": 4,
    "SeniorCitizen": 0,
    "Partner": 0,
    "Dependents": 0
}])

prediction = best_model.predict(sample_customer)[0]
probability = best_model.predict_proba(sample_customer)[0][1]

print("\nSample Customer Prediction:")
print("Prediction:", "Churn" if prediction == 1 else "No Churn")
print("Churn Probability:", round(float(probability), 4))

Dataset shape: (1000, 11)

First 5 rows:
   Tenure  MonthlyCharges  TotalCharges    ContractType InternetService  \
0       7          105.83        797.14        One year     Fiber optic   
1      55           72.29       3987.30        One year             DSL   
2      47           84.52       4005.41  Month-to-month     Fiber optic   
3      32           68.58       2272.33  Month-to-month             DSL   
4      31           65.74       1956.99  Month-to-month             DSL   

      PaymentMethod  SupportCalls  SeniorCitizen  Partner  Dependents  Churn  
0  Electronic check             3              0        1           1      1  
1     Bank transfer             0              0        1           1      0  
2       Credit card             4              1        0           0      1  
3  Electronic check             3              0        1           0      1  
4      Mailed check             1              0        0           1      0  

Churn distribution:
Churn
1    65